# Stage 4 QLoRA Training on Colab
Notebook pentru antrenare reală a adapter-ului LoRA. Implicit: Mistral 7B Instruct (`configs/qlora_mistral7b_instruct.yaml`). Pentru gpt-oss folosește `configs/qlora_colab_t4.yaml` (seq mai scurt).


## 0) Runtime
Seteaza runtime: **GPU (T4 sau A100)**.


In [ ]:
!nvidia-smi


In [ ]:
REPO_URL = "https://github.com/alexandraraluca/Hint_generator_Vmchecker.git"  # TODO
PROJECT_DIR = "/content/licenta_vmchecker"

!rm -rf $PROJECT_DIR
!git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR


In [ ]:
# Stack recomandat (Mistral 7B + opțional gpt-oss în același mediu):
#  - transformers 4.55+ (gpt-oss + Harmony; Mistral e suportat de versiuni recente)
#  - tokenizers 0.20+
#  - accelerate 1.x
# IMPORTANT: NU folosi transformers 5.x dacă încă testezi accelerate/compat.

!pip uninstall -y transformers tokenizers accelerate peft trl bitsandbytes
!pip install --no-cache-dir \
    "transformers>=4.55,<5.0" \
    "tokenizers>=0.20,<0.22" \
    "accelerate>=1.0,<2.0" \
    "peft>=0.13,<0.20" \
    "trl>=0.11,<0.25" \
    "bitsandbytes>=0.43" \
    "datasets>=2.20" \
    "sentencepiece" \
    "pyyaml" "orjson"

print("\nDONE installing. Acum: Runtime -> Restart runtime, apoi rerulează celulele de la 'Upload dataset' în jos.")

In [ ]:
# Verificare versiuni (rulează DUPĂ Restart runtime).
# Trebuie să vezi: transformers 4.55+, tokenizers 0.20+, accelerate 1.x, peft 0.13+
import importlib, importlib.metadata as md

for pkg in ["transformers", "tokenizers", "accelerate", "peft", "trl", "bitsandbytes", "datasets"]:
    try:
        v = md.version(pkg)
        print(f"  {pkg:14s} {v}")
    except md.PackageNotFoundError:
        print(f"  {pkg:14s} <not installed>")

import torch
print(f"\n  torch          {torch.__version__} | cuda available: {torch.cuda.is_available()} | device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
from google.colab import files
import os, shutil

target = f"{PROJECT_DIR}/data/hints"
os.makedirs(target, exist_ok=True)
uploaded = files.upload()  # finetune_train.jsonl, finetune_val.jsonl (+ optional finetune_test.jsonl)

for fn in uploaded.keys():
    shutil.move(fn, os.path.join(target, fn))

print(sorted(os.listdir(target)))


In [ ]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONPATH"] = PROJECT_DIR

!python -m src.stage4_finetune.train_qlora --config configs/qlora_mistral7b_instruct.yaml --dry-run


In [ ]:
!python -m src.stage4_finetune.train_qlora --config configs/qlora_mistral7b_instruct.yaml


In [ ]:
from google.colab import files

ARCHIVE = "/content/mistral7b_instruct_pa_hints_adapter.tgz"
!tar -czf $ARCHIVE -C $PROJECT_DIR/models mistral7b_instruct_pa_hints
!ls -lh $ARCHIVE
files.download(ARCHIVE)
